# Assessment 3 - Regulatory DQ, Lineage & Executive Dashboard

See `docs/milestones.md` and `docs/design/assignment.md` for task scope.
Connectivity conventions: see `00_template_connectivity_check.ipynb`.


In [1]:
import os
from pyspark.sql import SparkSession
import psycopg2

POSTGRES_DB = os.environ["POSTGRES_DB"]
POSTGRES_USER = os.environ["POSTGRES_USER"]
POSTGRES_PASSWORD = os.environ["POSTGRES_PASSWORD"]


## Task 1 - Profile Transaction Banking Data

Profile `source.payment_transactions` and `bronze.payment_transactions` for record-level quality,
reference integrity against `bronze.customer_master`, and distributional shape, then assess how each
check would run efficiently in Spark/Databricks against a much larger table.


In [2]:
spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("assessment3-task1-profiling")
    .getOrCreate()
)


def jdbc_table(table_name):
    return spark.read.jdbc(
        url=f"jdbc:postgresql://postgres:5432/{POSTGRES_DB}",
        table=table_name,
        properties={
            "user": POSTGRES_USER,
            "password": POSTGRES_PASSWORD,
            "driver": "org.postgresql.Driver",
        },
    )


source_df = jdbc_table("source.payment_transactions").cache()
bronze_df = jdbc_table("bronze.payment_transactions").cache()
customer_df = jdbc_table("bronze.customer_master").cache()

source_df.createOrReplaceTempView("payment_transactions_source")
bronze_df.createOrReplaceTempView("payment_transactions_bronze")
customer_df.createOrReplaceTempView("customer_master")

source_count = source_df.count()
bronze_count = bronze_df.count()
customer_count = customer_df.count()
date_range = source_df.selectExpr("min(payment_date) AS min_date", "max(payment_date) AS max_date").collect()[0]

print(f"[INFO] row counts: source={source_count} bronze={bronze_count} customer_master={customer_count}")
print(f"[INFO] payment_date range: {date_range['min_date']} to {date_range['max_date']}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/14 08:13:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[INFO] row counts: source=2005 bronze=2025 customer_master=315
[INFO] payment_date range: 2026-08-17 to 2026-08-21


### Record-level and reference-integrity checks

Duplicate `payment_id` is measured against the source extract - Bronze's own business key is
`(payment_id, batch_id)`, so a Bronze-level repeat is a reload signal, left to the reload-detection
investigation. Missing customer reference and inactive customer reference are two distinct
populations: the first has no `customer_master` row for the `customer_id` at all, the second has
history but none of it currently open (`effective_end_date IS NULL`). Multiple active customer
records looks for overlapping effective-dated windows for the same `customer_id`.


In [3]:
from pyspark.sql.functions import col, trim

VALID_STATUSES = ["COMPLETED", "PENDING", "FAILED", "REJECTED"]

duplicate_payment_id = spark.sql('''
    SELECT payment_id, COUNT(*) AS n
    FROM payment_transactions_source
    GROUP BY payment_id
    HAVING COUNT(*) > 1
''')
duplicate_payment_id_groups = duplicate_payment_id.count()

missing_customer_id = source_df.filter(col("customer_id").isNull() | (trim(col("customer_id")) == ""))
missing_customer_id_count = missing_customer_id.count()

invalid_status = source_df.filter(~col("status").isin(VALID_STATUSES))
invalid_status_count = invalid_status.count()

missing_currency = source_df.filter(col("currency").isNull() | (trim(col("currency")) == ""))
missing_currency_count = missing_currency.count()

# ISO 3166-1 alpha-2 shape check only - no country-code lookup table is seeded, so a
# syntactically valid but non-existent code passes this check; only a malformed value fails it.
invalid_beneficiary_country = source_df.filter(
    col("beneficiary_country").isNull() | ~col("beneficiary_country").rlike("^[A-Z]{2}$")
)
invalid_beneficiary_country_count = invalid_beneficiary_country.count()

negative_or_zero_amount = source_df.filter(col("amount") <= 0)
negative_or_zero_amount_count = negative_or_zero_amount.count()

missing_customer_reference = spark.sql('''
    SELECT p.payment_id, p.customer_id
    FROM payment_transactions_bronze p
    WHERE NOT EXISTS (SELECT 1 FROM customer_master c WHERE c.customer_id = p.customer_id)
''')
missing_customer_reference_count = missing_customer_reference.count()

inactive_customer_reference = spark.sql('''
    SELECT p.payment_id, p.customer_id
    FROM payment_transactions_bronze p
    WHERE EXISTS (SELECT 1 FROM customer_master c WHERE c.customer_id = p.customer_id)
      AND NOT EXISTS (
        SELECT 1 FROM customer_master c
        WHERE c.customer_id = p.customer_id AND c.effective_end_date IS NULL
      )
''')
inactive_customer_reference_count = inactive_customer_reference.count()

overlapping_customer_records = spark.sql('''
    SELECT DISTINCT a.customer_id
    FROM customer_master a
    JOIN customer_master b
      ON  a.customer_id = b.customer_id
      AND a.effective_start_date < b.effective_start_date
      AND a.effective_start_date <= COALESCE(b.effective_end_date, DATE '9999-12-31')
      AND COALESCE(a.effective_end_date, DATE '9999-12-31') >= b.effective_start_date
''')
overlapping_customer_records_count = overlapping_customer_records.count()

print("[INFO] record-level and reference-integrity checks:")
print(f"  duplicate payment_id: {duplicate_payment_id_groups} groups")
print(f"  missing customer_id: {missing_customer_id_count} rows")
print(f"  invalid payment status: {invalid_status_count} rows")
print(f"  missing currency: {missing_currency_count} rows")
print(f"  invalid beneficiary country: {invalid_beneficiary_country_count} rows")
print(f"  negative or zero amount: {negative_or_zero_amount_count} rows")
print(f"  missing customer reference record: {missing_customer_reference_count} rows")
print(f"  payment linked to inactive customer record: {inactive_customer_reference_count} rows")
print(f"  multiple active customer records: {overlapping_customer_records_count} customers")


[INFO] record-level and reference-integrity checks:
  duplicate payment_id: 5 groups
  missing customer_id: 4 rows
  invalid payment status: 0 rows
  missing currency: 0 rows
  invalid beneficiary country: 2 rows
  negative or zero amount: 6 rows
  missing customer reference record: 27 rows
  payment linked to inactive customer record: 33 rows
  multiple active customer records: 20 customers


### Distributional checks

Daily transaction-volume spikes are flagged where a day's count sits more than two standard
deviations from the seeded period's own mean. Payment-channel distribution is flagged where a
channel's daily share of volume deviates more than 15 percentage points from its own period-average
share. Cross-border classification anomalies are rows the domestic/cross-border test cannot evaluate
at all, or rows where case or whitespace noise in the country codes alone produces a mismatched
result.


In [4]:
volume_by_day = spark.sql('''
    WITH daily AS (
      SELECT payment_date, COUNT(*) AS n FROM payment_transactions_bronze GROUP BY payment_date
    ),
    stats AS (SELECT AVG(n) AS mean_n, STDDEV_SAMP(n) AS sd_n FROM daily)
    SELECT d.payment_date, d.n, (d.n - s.mean_n) / NULLIF(s.sd_n, 0) AS z_score
    FROM daily d CROSS JOIN stats s
    ORDER BY d.payment_date
''')
volume_spikes = volume_by_day.filter("ABS(z_score) > 2")
volume_spike_days = volume_spikes.count()

channel_share = spark.sql('''
    SELECT payment_date, payment_channel,
           COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY payment_date) AS daily_share
    FROM payment_transactions_bronze
    GROUP BY payment_date, payment_channel
''')
channel_share.createOrReplaceTempView("channel_share")
channel_baseline = spark.sql('''
    SELECT cs.payment_date, cs.payment_channel, cs.daily_share, b.baseline_share
    FROM channel_share cs
    JOIN (SELECT payment_channel, AVG(daily_share) AS baseline_share FROM channel_share GROUP BY payment_channel) b
      USING (payment_channel)
''')
channel_outliers = channel_baseline.filter("ABS(daily_share - baseline_share) > 0.15")
channel_outlier_rows = channel_outliers.count()

channel_totals = bronze_df.groupBy("payment_channel").count().orderBy("payment_channel")

cross_border_anomalies = spark.sql('''
    SELECT p.payment_id, c.residence_country, p.beneficiary_country
    FROM payment_transactions_bronze p
    LEFT JOIN customer_master c
      ON  c.customer_id = p.customer_id
      AND p.payment_date BETWEEN c.effective_start_date AND COALESCE(c.effective_end_date, DATE '9999-12-31')
    WHERE c.residence_country IS NULL
       OR (UPPER(TRIM(c.residence_country)) = UPPER(TRIM(p.beneficiary_country))
           AND c.residence_country <> p.beneficiary_country)
''')
cross_border_anomaly_count = cross_border_anomalies.count()

print("[INFO] distributional checks:")
print(f"  daily volume outside 2 standard deviations: {volume_spike_days} of {volume_by_day.count()} days")
volume_by_day.show(20, truncate=False)
print(f"  channel-share deviation beyond 15pp: {channel_outlier_rows} day/channel combinations")
print("  channel volume distribution:")
channel_totals.show(truncate=False)
print(f"  cross-border classification anomalies: {cross_border_anomaly_count} rows")


[INFO] distributional checks:


  daily volume outside 2 standard deviations: 0 of 5 days


+------------+---+-------------------+
|payment_date|n  |z_score            |
+------------+---+-------------------+
|2026-08-17  |400|-0.5555555555555556|
|2026-08-18  |400|-0.5555555555555556|
|2026-08-19  |421|1.7777777777777777 |
|2026-08-20  |402|-0.3333333333333333|
|2026-08-21  |402|-0.3333333333333333|
+------------+---+-------------------+

  channel-share deviation beyond 15pp: 0 day/channel combinations
  channel volume distribution:


+---------------+-----+
|payment_channel|count|
+---------------+-----+
|API            |492  |
|BRANCH         |515  |
|INTERNET       |550  |
|MOBILE         |468  |
+---------------+-----+

  cross-border classification anomalies: 60 rows


### Spark-scale handling at production volume

Every check above runs as a full in-memory scan against the seeded volume budget. At the
assignment's production scale, the same logic holds without a design change, applying the following
techniques:

- **partitioned scans** - `payment_transactions` partitioned by `payment_date` lets every check above
  that filters or groups on the date column skip partitions outside its window, avoiding a full table
  scan.
- **approximate cardinality** - the duplicate-`payment_id` and overlapping-customer-record checks
  only need an exact count where the group size is small (an actual duplicate group); a first-pass
  `approx_count_distinct` on `payment_id` cheaply confirms whether any duplication exists at all
  before paying for the exact `GROUP BY ... HAVING COUNT(*) > 1`.
- **broadcast joins** - `bronze.customer_master` is small relative to `bronze.payment_transactions`;
  broadcasting it (`F.broadcast(customer_df)`) on every reference-integrity and cross-border check
  avoids shuffling the much larger payment table to perform the join.
- **column pruning** - every check above reads only the columns it filters, groups, or joins on, a
  subset of the full schema carried through automatically by Spark's own predicate/column pushdown
  against the DataFrames loaded here.


In [5]:
print(
    "[PASS] task 1 profiling: record-level, reference-integrity, and distributional checks complete"
)
spark.stop()


[PASS] task 1 profiling: record-level, reference-integrity, and distributional checks complete
